# Deep Learning Individual Project — RMSprop Variant

This cloned notebook (`submission_s3343711_rmsprop.ipynb`) is used to
experiment with an alternative CNN model that uses the **RMSprop**
optimizer and a slightly different dense stack, while keeping the rest
of the pipeline (EDA, preprocessing, train/validation split) aligned
with the main submission notebook.


> Note: The main graded notebook remains `submission_s3343711.ipynb`.
> This clone is only for comparing an RMSprop-based model against
> the Adam-based baseline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Load dataset
dataset = np.load("4class_32x32.npz")
X = dataset["X"]
y = dataset["y"]

# Stratified train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

# Min–max normalization using training set statistics
train_min = X_train.min()
train_max = X_train.max()

X_train_norm = (X_train - train_min) / (train_max - train_min)
X_val_norm = (X_val - train_min) / (train_max - train_min)

# Add channel dimension
X_train_final = X_train_norm.reshape(-1, 32, 32, 1)
X_val_final = X_val_norm.reshape(-1, 32, 32, 1)

print("Train shape:", X_train_final.shape)
print("Val shape:", X_val_final.shape)

In [ ]:
# RMSprop-based CNN model
rmsprop_model = keras.Sequential([
    layers.Input(shape=(32, 32, 1)),
    
    # Conv blocks (same as baseline)
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    # Alternative dense stack
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(4, activation="softmax"),
])

rmsprop_model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

rmsprop_model.summary()

RMSPROP_EPOCHS = 10
RMSPROP_BATCH_SIZE = 32

rmsprop_history = rmsprop_model.fit(
    X_train_final, y_train,
    validation_data=(X_val_final, y_val),
    epochs=RMSPROP_EPOCHS,
    batch_size=RMSPROP_BATCH_SIZE,
    verbose=1,
)

print("Final Training Accuracy (RMSprop):", rmsprop_history.history["accuracy"][-1])
print("Final Validation Accuracy (RMSprop):", rmsprop_history.history["val_accuracy"][-1])